## Creating an a summary agent for control performance.
### Approach to be used
- Create an endpoint for a table that will store the control summary
- An agent that will retrive control exceptions and relevant context using RAG.
- An agent should review those exceptions and record an insightful summary.
- An agent should review that summary and determine if it sufficient to be recorded.

Firstly, create and endpoint to be used to store agent feedback

Import all necessary libraries

In [16]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner,trace, function_tool
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
import asyncio
import httpx
from typing import Any
from agent_config import system_prompt 
from pydantic import BaseModel,Field
from typing import Optional, List
import requests
import json
load_dotenv(override=True)

True

Structured Outputs

In [38]:
class ControlSummary(BaseModel):
    index_number: int =Field(description='Generate a random integer between 1 and 1000')
    analysis_date: str = Field(description='Date of the analysis (The current date)')
    control_summary: str= Field(description='Sentences that describe exceptions, including the exception count, value, similarity and abnomarlity')
    issue_resolution_status: Optional[str] = Field(description='Write "Pending" for this field')
    feedback_receiving_date: Optional[str] = Field(description='Write "Pending" for this field')
    outcome: Optional[str] = Field(description='Write "Pending" for this field')
    day: Optional[str] = Field(description='Write "Pending" for this field')

Create a client for the anthropic's API

In [3]:
anthropic_client = AsyncOpenAI(
    api_key= os.getenv(key='ANTHROPIC_API_KEY'),
    base_url='https://api.anthropic.com/v1/',
)

Create the anthropic model

In [4]:
anthropic_model = OpenAIChatCompletionsModel(
                    model="claude-haiku-4-5",
                    openai_client=anthropic_client,
                    )
openai_model = "gpt-4o-mini"

Create an agent fetch detail tool

In [5]:
ALL_ENDPOINTS = ["/data/exception", "/data/logic", "/data/dictionary"]


async def _fetch_one(client: httpx.AsyncClient, endpoint: str) -> tuple[str, Any]:
    BASE_URL = "https://controldev-apfxc7h7etf4breb.southafricanorth-01.azurewebsites.net"
    key = endpoint.split("/")[-1]
    try:
        response = await client.get(BASE_URL + endpoint)
        if response.status_code == 200:
            return key, response.json()
    except httpx.RequestError:
        pass
    return key, None

@function_tool
async def fetch_controlweb_data(endpoints: list[str] | None = None) -> dict[str, Any]:
    """
    Fetch ControlWeb data from one or more endpoints concurrently.

    Args:
        endpoints: Subset of ['/data/exception', '/data/logic', '/data/dictionary'].
                   Defaults to all three if not provided.

    Returns:
        Dict keyed by endpoint name ('exception', 'logic', 'dictionary').
        Keys for failed/non-200 requests are omitted.
    """
    targets = endpoints if endpoints is not None else ALL_ENDPOINTS
    async with httpx.AsyncClient() as client:
        tasks = [_fetch_one(client, ep) for ep in targets]
        results = await asyncio.gather(*tasks)
    return {key: data for key, data in results if data is not None}

In [6]:
Rewrite_data = Agent(name="AI assistant",
                      instructions=system_prompt.receipent_agent_prompt,
                      tools=[fetch_controlweb_data],
                      model= openai_model,
                      )

Convert the rewrite agent to a tool

In [7]:
rewrite_tool = Rewrite_data.as_tool(tool_name="rewrite_tool", tool_description="an AI assistant that rewrites JSON files into easily processable output for downstream agents")

Create the reviewing agent

In [39]:
fraud_analyst = Agent(name="Fraud analyst",
                      instructions=system_prompt.processing_agent_prompt,
                      tools=[rewrite_tool],
                      model=anthropic_model,
                      output_type=ControlSummary
                      )

create the user message for the agent

In [9]:
message = f"Call the rewrite_data tool to extract the information to review, provide the tool with following list of Endpoints: {ALL_ENDPOINTS}"

In [44]:
with trace("Multi-agent outcome for reviewing data"):
    result = await Runner.run(fraud_analyst,message)

In [45]:
print(result.final_output)

index_number=847 analysis_date='2026-04-02' control_summary="Control 1 (Check suspended customers) identified 2 exceptions. Both records feature accounts with 'Suspended' status and elevated usage amounts (90.0 and 200.0 USD, well above the normal 10-100 USD range documented in the data dictionary). Exception 1: Charles Hernandez (USR010, ACC010) - suspended since unknown date with 90.0 USD usage; Exception 2: Isaac Harris (USR042, ACC042) - suspended since unknown date with 200.0 USD usage (220% above median expected usage). The control logic executes a straightforward status-based query, and both exceptions are direct matches. Anomalies include the suspension of accounts without documented reasons and usage amounts that exceed normal patterns, suggesting potential fraud risk or account compromise. Detection timestamp for both exceptions is 2026-04-02T16:40:09.087326." issue_resolution_status='Pending' feedback_receiving_date='Pending' outcome='Pending' day='Pending'


convert to a dictionary

In [46]:
output_results: ControlSummary = result.final_output

output_dictionary = output_results.model_dump()

In [47]:
print(type(output_dictionary))

<class 'dict'>


needed this part for testing on Fast api docs

In [ ]:
#json_string= json.dumps(output_dictionary)
#print(json_string)

{"index_number": 347, "analysis_date": "2026-04-08", "control_summary": "The control flagged 2 exception records designed to identify suspended customer accounts. Both exceptions involve customers (Charles Hernandez - USR010 and Isaac Harris - USR042) with Suspended account status. Charles Hernandez has a usage amount of $90.0 (within normal 10-100 range), while Isaac Harris shows elevated usage of $200.0 (above normal range). Both accounts were registered in 2019 and 2021 respectively and were detected on 2026-04-02. Anomaly indicators include the Suspended status without documented reasons, and Isaac Harris's usage amount ($200.0) exceeds the normal pattern threshold of 10-100, representing a potential outlier or high-usage anomaly requiring further investigation.", "issue_resolution_status": "Pending", "feedback_receiving_date": "Pending", "outcome": "Pending", "day": "Pending"}


call the summary API ingestion endpoint

In [48]:
end_point = "/insert/control_summary"
BASE_URL = "https://controldev-apfxc7h7etf4breb.southafricanorth-01.azurewebsites.net"
URL = BASE_URL+end_point
response = requests.post(url=URL,json=[output_dictionary])

In [22]:
print(URL)

https://controldev-apfxc7h7etf4breb.southafricanorth-01.azurewebsites.net/insert/control_summary


In [49]:
print(response)

<Response [201]>


send to supabase